<a href="https://www.kaggle.com/code/simarbirsinghsandhu/catboost-gpu-pb-0-95017?scriptVersionId=319084322" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🏎️ F1 Pit Stop Prediction | CatBoost GPU Baseline
### Playground Series S6E5

**🏆 OOF ROC-AUC: 0.960292 | 📊 Public LB: 0.95017**

⚡ CatBoost + Optuna (GPU) | 5-Fold Stratified CV

**Why CatBoost over XGBoost here:**
- Handles `Driver`, `Compound`, `Race` natively — no label encoding needed
- Built-in ordered boosting reduces overfitting on synthetic data
- Different error patterns from XGBoost → blend both for higher LB

> Upvote if useful 🙌 | [RealMLP](https://www.kaggle.com/code/simarbirsinghsandhu/realmlp-encoding-fe) (PB 0.95253) | [XGBoost GPU Baseline](https://www.kaggle.com/code/simarbirsinghsandhu/xgboost-gpu-fe-encoding) | [Full EDA](https://www.kaggle.com/code/simarbirsinghsandhu/s6e5-detailed-eda-original-dataset-analysis)

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import catboost as cb
import optuna
import matplotlib.pyplot as plt
import warnings

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"catboost {cb.__version__}")

catboost 1.2.10


## 2. Config

In [2]:
CONFIG = {
    "train_path"       : "/kaggle/input/competitions/playground-series-s6e5/train.csv",
    "test_path"        : "/kaggle/input/competitions/playground-series-s6e5/test.csv",
    "orig_path"        : "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
    "target_col"       : "PitNextLap",
    "id_col"           : "id",
    "n_folds"          : 5,
    "seed"             : 42,
    "optuna_trials"    : 50,
    "optuna_folds"     : 3,
    "optuna_subsample" : 200_000,
}

## 3. Load Data

In [3]:
train = pd.read_csv(CONFIG["train_path"])
test  = pd.read_csv(CONFIG["test_path"])
orig  = pd.read_csv(CONFIG["orig_path"])

orig.drop(columns=["Normalized_TyreLife"], inplace=True, errors="ignore")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")
print(f"\nTarget distribution:")
print(train["PitNextLap"].value_counts(normalize=True).round(4))

Train : (439140, 16)
Test  : (188165, 15)
Orig  : (101371, 15)

Target distribution:
PitNextLap
0.0    0.801
1.0    0.199
Name: proportion, dtype: float64


## 4. Feature Engineering

Same features as the XGBoost baseline — built from EDA findings.  
Key difference: `Driver`, `Compound`, `Race` passed as **categorical features** to CatBoost directly. No label encoding needed — CatBoost handles them natively with its own ordered target statistics, which is more principled than simple label encoding.

In [4]:
COMPOUND_STINT_MEDIANS = {
    "SOFT"        : 14.0,
    "MEDIUM"      : 17.0,
    "HARD"        : 23.0,
    "INTERMEDIATE": 16.0,
    "WET"         : 14.0,
}

COMPOUND_HARDNESS = {
    "SOFT"        : 1,
    "MEDIUM"      : 2,
    "HARD"        : 3,
    "INTERMEDIATE": 1,
    "WET"         : 0,
}

def engineer_features(df):
    df = df.copy()

    df["ExpectedStint"]       = df["Compound"].map(COMPOUND_STINT_MEDIANS).fillna(17.0)
    df["TyreLife_Normalized"] = df["TyreLife"] / df["ExpectedStint"]
    df["TyreLife_sq"]         = df["TyreLife"] ** 2
    df["TyreLife_sqrt"]       = np.sqrt(df["TyreLife"])
    df["TyreLife_log1p"]      = np.log1p(df["TyreLife"])
    df["Compound_Hardness"]   = df["Compound"].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    df["TyreLife_x_Hardness"] = df["TyreLife"] * df["Compound_Hardness"]
    df["Norm_x_Hardness"]     = df["TyreLife_Normalized"] * df["Compound_Hardness"]
    df["Is_Fresh"]            = (df["TyreLife"] <= 3).astype(np.int8)
    df["Is_Old"]              = (df["TyreLife"] > 20).astype(np.int8)
    df["Is_VeryOld"]          = (df["TyreLife"] > 40).astype(np.int8)
    df["Is_FirstStint"]       = (df["Stint"] == 1).astype(np.int8)
    df["DegRate"]             = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)
    df["Early_Race"]          = (df["RaceProgress"] < 0.25).astype(np.int8)
    df["Late_Race"]           = (df["RaceProgress"] >= 0.75).astype(np.int8)
    df["VeryLate_Race"]       = (df["RaceProgress"] >= 0.90).astype(np.int8)
    df["Is_2025"]             = (df["Year"] == 2025).astype(np.int8)

    return df

train = engineer_features(train)
test  = engineer_features(test)
orig  = engineer_features(orig)

print(f"Engineered 16 new features")

Engineered 16 new features


## 5. Preprocessing

In [5]:
CAT_COLS  = ["Driver", "Compound", "Race"]
DROP_COLS = ["id", "PitNextLap", "ExpectedStint"]

# Convert categoricals to string for CatBoost
for col in CAT_COLS:
    train[col] = train[col].astype(str)
    test[col]  = test[col].astype(str)
    orig[col]  = orig[col].astype(str)

# Domain gap flag
train["is_original"] = 0
orig["is_original"]  = 1
test["is_original"]  = 0

FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

# Align original
orig_aligned = orig.reindex(columns=FEATURE_COLS + ["PitNextLap"], fill_value=0)
orig_aligned["PitNextLap"] = orig_aligned["PitNextLap"].astype(float)
train_full = pd.concat([train, orig_aligned], ignore_index=True)

X_df      = train_full[FEATURE_COLS]
y         = train_full["PitNextLap"].values
X_test_df = test[FEATURE_COLS]

# CatBoost needs categorical feature indices
cat_feature_indices = [X_df.columns.get_loc(c) for c in CAT_COLS]

print(f"Features         : {len(FEATURE_COLS)}")
print(f"Train rows       : {len(X_df):,}")
print(f"Positive rate    : {y.mean():.4f}")
print(f"Cat feature idx  : {cat_feature_indices}")

Features         : 31
Train rows       : 540,511
Positive rate    : 0.2094
Cat feature idx  : [0, 1, 2]


## 6. Optuna Hyperparameter Search

3-fold CV on 200K subsample, optimizing ROC-AUC directly.  
CatBoost search space differs from XGBoost — `depth`, `l2_leaf_reg`, `bagging_temperature` are the key levers.

In [6]:
def run_optuna(X_df, y, cat_indices, cfg):
    n_sub = cfg["optuna_subsample"]
    rng   = np.random.RandomState(cfg["seed"])
    idx   = rng.choice(len(X_df), min(n_sub, len(X_df)), replace=False)
    X_opt = X_df.iloc[idx].reset_index(drop=True)
    y_opt = y[idx]
    print(f"Subsampled to {len(X_opt):,} rows for Optuna")

    kf = StratifiedKFold(n_splits=cfg["optuna_folds"],
                         shuffle=True, random_state=cfg["seed"])

    def objective(trial):
        params = {
            "iterations"          : 3000,
            "learning_rate"       : trial.suggest_float("learning_rate",       0.01,  0.15, log=True),
            "depth"               : trial.suggest_int(  "depth",               4,     8),
            "l2_leaf_reg"         : trial.suggest_float("l2_leaf_reg",         1.0,   10.0, log=True),
            "bagging_temperature" : trial.suggest_float("bagging_temperature", 0.0,   1.0),
            "random_strength"     : trial.suggest_float("random_strength",     0.0,   2.0),
            "border_count"        : trial.suggest_int(  "border_count",        32,    255),
            "scale_pos_weight"    : trial.suggest_float("scale_pos_weight",    1.0,   8.0),
            "task_type"           : "GPU",
            "eval_metric"         : "AUC",
            "loss_function"       : "Logloss",
            "random_seed"         : cfg["seed"],
            "verbose"             : False,
        }

        aucs = []
        for tr_idx, va_idx in kf.split(X_opt, y_opt):
            X_tr = X_opt.iloc[tr_idx]
            X_va = X_opt.iloc[va_idx]
            y_tr = y_opt[tr_idx]
            y_va = y_opt[va_idx]

            train_pool = cb.Pool(X_tr, y_tr, cat_features=cat_indices)
            val_pool   = cb.Pool(X_va, y_va, cat_features=cat_indices)

            model = cb.CatBoostClassifier(**params)
            model.fit(train_pool,
                      eval_set=val_pool,
                      early_stopping_rounds=100,
                      verbose=False)

            aucs.append(roc_auc_score(y_va,
                        model.predict_proba(X_va)[:, 1]))
        return np.mean(aucs)

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=cfg["seed"]))
    study.optimize(objective, n_trials=cfg["optuna_trials"], show_progress_bar=True)

    print(f"\n✅ Best ROC-AUC : {study.best_value:.6f}")
    print(f"Best params    :\n{study.best_params}")
    return study.best_params, study
    
# Uncomment to run optuna tune
# best_params, study = run_optuna(X_df, y, cat_feature_indices, CONFIG)

## 7. Best params
These are auto-filled by Optuna above. If you want to skip Optuna on re-runs, comment out the cell above and paste your best params here.

In [7]:
# best_params is already set by Optuna above
# If you skipped Optuna, uncomment and paste your params

best_params = {
    "learning_rate"      : 0.018,
    "depth"              : 8,
    "l2_leaf_reg"        : 8.5,
    "random_strength"    : 0.65,
    "bootstrap_type"     : "Bayesian",
    "bagging_temperature": 0.45,
    "auto_class_weights" : "Balanced",
    "eval_metric"        : "AUC",
}

print("Using params:", best_params)

Using params: {'learning_rate': 0.018, 'depth': 8, 'l2_leaf_reg': 8.5, 'random_strength': 0.65, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.45, 'auto_class_weights': 'Balanced', 'eval_metric': 'AUC'}


## 8. OOF Training

Full 5-fold stratified CV. CatBoost Pool objects handle categorical features properly inside each fold.

In [8]:
def catboost_oof(X_df, y, X_test_df, cat_indices, params, cfg):
    kf         = StratifiedKFold(n_splits=cfg["n_folds"],
                                 shuffle=True, random_state=cfg["seed"])
    oof_preds  = np.zeros(len(X_df))
    test_preds = np.zeros(len(X_test_df))
    fold_aucs  = []

    run_params = {
        "iterations"          : 15000,
        "task_type"           : "GPU",
        "loss_function"       : "Logloss",
        "random_seed"         : cfg["seed"],
        "allow_writing_files" : False,
        "verbose"             : False,
        **params,
    }

    test_pool = cb.Pool(X_test_df, cat_features=cat_indices)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_df, y)):
        X_tr = X_df.iloc[tr_idx]
        X_va = X_df.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        train_pool = cb.Pool(X_tr, y_tr, cat_features=cat_indices)
        val_pool   = cb.Pool(X_va, y_va, cat_features=cat_indices)

        model = cb.CatBoostClassifier(**run_params)
        model.fit(train_pool,
                  eval_set=val_pool,
                  early_stopping_rounds=500,
                  verbose=500)

        val_preds         = model.predict_proba(X_va)[:, 1]
        oof_preds[va_idx] = val_preds
        test_preds       += model.predict_proba(X_test_df)[:, 1] / cfg["n_folds"]
        auc               = roc_auc_score(y_va, val_preds)
        fold_aucs.append(auc)
        print(f"Fold {fold+1} AUC: {auc:.6f}\n")

    oof_auc = roc_auc_score(y, oof_preds)
    print("─" * 45)
    for i, s in enumerate(fold_aucs):
        print(f"  Fold {i+1}: {s:.6f}")
    print("─" * 45)
    print(f"  Mean  : {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}")
    print(f"  OOF   : {oof_auc:.6f}")
    print("─" * 45)

    return oof_preds, test_preds, fold_aucs, oof_auc, model

oof_preds, test_preds, fold_aucs, oof_auc, last_model = catboost_oof(
    X_df, y, X_test_df, cat_feature_indices, best_params, CONFIG
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8911460	best: 0.8911460 (0)	total: 6.71s	remaining: 1d 3h 57m 54s
500:	test: 0.9457782	best: 0.9457782 (500)	total: 34.3s	remaining: 16m 32s
1000:	test: 0.9518935	best: 0.9518935 (1000)	total: 1m 1s	remaining: 14m 20s
1500:	test: 0.9544612	best: 0.9544612 (1500)	total: 1m 28s	remaining: 13m 19s
2000:	test: 0.9559554	best: 0.9559554 (2000)	total: 1m 56s	remaining: 12m 35s
2500:	test: 0.9568431	best: 0.9568431 (2500)	total: 2m 23s	remaining: 11m 58s
3000:	test: 0.9575048	best: 0.9575048 (3000)	total: 2m 51s	remaining: 11m 24s
3500:	test: 0.9580467	best: 0.9580467 (3500)	total: 3m 19s	remaining: 10m 54s
4000:	test: 0.9584400	best: 0.9584404 (3999)	total: 3m 47s	remaining: 10m 24s
4500:	test: 0.9587950	best: 0.9587950 (4500)	total: 4m 15s	remaining: 9m 55s
5000:	test: 0.9590969	best: 0.9590973 (4997)	total: 4m 43s	remaining: 9m 25s
5500:	test: 0.9593313	best: 0.9593313 (5499)	total: 5m 11s	remaining: 8m 57s
6000:	test: 0.9595045	best: 0.9595045 (6000)	total: 5m 40s	remaining: 8m

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8997311	best: 0.8997311 (0)	total: 60.3ms	remaining: 15m 4s
500:	test: 0.9429500	best: 0.9429500 (500)	total: 27.8s	remaining: 13m 23s
1000:	test: 0.9499515	best: 0.9499515 (1000)	total: 54.9s	remaining: 12m 48s
1500:	test: 0.9526358	best: 0.9526358 (1500)	total: 1m 22s	remaining: 12m 18s
2000:	test: 0.9543061	best: 0.9543061 (2000)	total: 1m 49s	remaining: 11m 52s
2500:	test: 0.9553050	best: 0.9553050 (2500)	total: 2m 17s	remaining: 11m 25s
3000:	test: 0.9560150	best: 0.9560150 (3000)	total: 2m 44s	remaining: 10m 58s
3500:	test: 0.9565777	best: 0.9565777 (3500)	total: 3m 12s	remaining: 10m 32s
4000:	test: 0.9569924	best: 0.9569924 (4000)	total: 3m 40s	remaining: 10m 6s
4500:	test: 0.9573403	best: 0.9573403 (4500)	total: 4m 8s	remaining: 9m 40s
5000:	test: 0.9576291	best: 0.9576291 (5000)	total: 4m 36s	remaining: 9m 13s
5500:	test: 0.9578812	best: 0.9578812 (5500)	total: 5m 5s	remaining: 8m 47s
6000:	test: 0.9581193	best: 0.9581199 (5998)	total: 5m 34s	remaining: 8m 20s
6500

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8889801	best: 0.8889801 (0)	total: 55.5ms	remaining: 13m 52s
500:	test: 0.9440037	best: 0.9440037 (500)	total: 27.8s	remaining: 13m 24s
1000:	test: 0.9503614	best: 0.9503614 (1000)	total: 54.8s	remaining: 12m 46s
1500:	test: 0.9531571	best: 0.9531571 (1500)	total: 1m 22s	remaining: 12m 18s
2000:	test: 0.9547569	best: 0.9547569 (2000)	total: 1m 49s	remaining: 11m 50s
2500:	test: 0.9558256	best: 0.9558256 (2500)	total: 2m 16s	remaining: 11m 23s
3000:	test: 0.9566171	best: 0.9566171 (3000)	total: 2m 44s	remaining: 10m 57s
3500:	test: 0.9571973	best: 0.9571973 (3500)	total: 3m 11s	remaining: 10m 30s
4000:	test: 0.9576787	best: 0.9576787 (4000)	total: 3m 39s	remaining: 10m 4s
4500:	test: 0.9580564	best: 0.9580564 (4500)	total: 4m 8s	remaining: 9m 38s
5000:	test: 0.9583311	best: 0.9583317 (4999)	total: 4m 36s	remaining: 9m 12s
5500:	test: 0.9585596	best: 0.9585596 (5500)	total: 5m 4s	remaining: 8m 45s
6000:	test: 0.9587850	best: 0.9587850 (6000)	total: 5m 32s	remaining: 8m 18s
650

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9000105	best: 0.9000105 (0)	total: 60.7ms	remaining: 15m 9s
500:	test: 0.9446070	best: 0.9446070 (500)	total: 27.6s	remaining: 13m 19s
1000:	test: 0.9512426	best: 0.9512426 (1000)	total: 54.2s	remaining: 12m 37s
1500:	test: 0.9539598	best: 0.9539598 (1500)	total: 1m 20s	remaining: 12m 6s
2000:	test: 0.9554768	best: 0.9554768 (2000)	total: 1m 47s	remaining: 11m 38s
2500:	test: 0.9564942	best: 0.9564942 (2500)	total: 2m 14s	remaining: 11m 12s
3000:	test: 0.9572170	best: 0.9572170 (3000)	total: 2m 41s	remaining: 10m 46s
3500:	test: 0.9577474	best: 0.9577474 (3500)	total: 3m 9s	remaining: 10m 20s
4000:	test: 0.9581311	best: 0.9581311 (4000)	total: 3m 36s	remaining: 9m 54s
4500:	test: 0.9584614	best: 0.9584614 (4500)	total: 4m 3s	remaining: 9m 28s
5000:	test: 0.9587837	best: 0.9587837 (5000)	total: 4m 31s	remaining: 9m 2s
5500:	test: 0.9589846	best: 0.9589846 (5500)	total: 4m 59s	remaining: 8m 37s
6000:	test: 0.9592113	best: 0.9592113 (6000)	total: 5m 27s	remaining: 8m 10s
6500:	

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8931982	best: 0.8931982 (0)	total: 54.9ms	remaining: 13m 43s
500:	test: 0.9460357	best: 0.9460357 (500)	total: 27.4s	remaining: 13m 14s
1000:	test: 0.9527864	best: 0.9527864 (1000)	total: 54.2s	remaining: 12m 38s
1500:	test: 0.9554905	best: 0.9554905 (1500)	total: 1m 21s	remaining: 12m 9s
2000:	test: 0.9569408	best: 0.9569408 (2000)	total: 1m 48s	remaining: 11m 41s
2500:	test: 0.9579287	best: 0.9579287 (2500)	total: 2m 14s	remaining: 11m 14s
3000:	test: 0.9586751	best: 0.9586751 (3000)	total: 2m 42s	remaining: 10m 48s
3500:	test: 0.9592013	best: 0.9592018 (3496)	total: 3m 9s	remaining: 10m 22s
4000:	test: 0.9595836	best: 0.9595836 (4000)	total: 3m 37s	remaining: 9m 56s
4500:	test: 0.9599262	best: 0.9599264 (4499)	total: 4m 4s	remaining: 9m 30s
5000:	test: 0.9602175	best: 0.9602175 (5000)	total: 4m 32s	remaining: 9m 4s
5500:	test: 0.9604602	best: 0.9604604 (5499)	total: 4m 59s	remaining: 8m 37s
6000:	test: 0.9606944	best: 0.9606944 (6000)	total: 5m 27s	remaining: 8m 11s
6500:

## 9. Submission

In [9]:
submission = pd.DataFrame({
    "id"        : test["id"],
    "PitNextLap": test_preds,
})
submission.to_csv("submission.csv", index=False)

print("✅ Submission saved")
print(f"   Shape  : {submission.shape}")
print(f"   Min    : {test_preds.min():.4f}")
print(f"   Max    : {test_preds.max():.4f}")
print(f"   Mean   : {test_preds.mean():.4f}")
print(f"\n   OOF AUC: {oof_auc:.6f}")
print(submission.head())

✅ Submission saved
   Shape  : (188165, 2)
   Min    : 0.0000
   Max    : 0.9986
   Mean   : 0.2708

   OOF AUC: 0.960608
       id  PitNextLap
0  439140    0.022819
1  439141    0.021382
2  439142    0.013387
3  439143    0.292397
4  439144    0.972658


## Summary

| | Score |
|---|---|
| OOF ROC-AUC | 0.960292 |
| Public LB | 0.95017 |

**Why CatBoost complements XGBoost:**
- Native categorical handling → Driver/Race/Compound encoded more carefully
- Ordered boosting → different bias-variance tradeoff
- Different prediction errors → blending both improves LB

**Next step — blend XGBoost + CatBoost + RealMLP OOFs:**

---
*Upvote if useful 🙌 | [RealMLP](https://www.kaggle.com/code/simarbirsinghsandhu/realmlp-encoding-fe) (PB 0.95253) | [XGBoost GPU Baseline](https://www.kaggle.com/code/simarbirsinghsandhu/xgboost-gpu-fe-encoding) | [Full EDA](https://www.kaggle.com/code/simarbirsinghsandhu/s6e5-detailed-eda-original-dataset-analysis)*